# 画像を切り取る

周辺環境の影響を無くすために、CAMERAディレクトリにある画像ファイルをすべてクロップして指定のディレクトリにコピーします。

電光掲示板の方向のデータセットに対しては、中央部だけを切り抜き、他は黒塗り

box = (0, 70, 224, 150)

５４０度ターンのデータセットに対しては、ギャラリー対策として、上部を黒塗りにし

box = (0, 45, 224, 224)

をコメントアウトで設定します。
    

In [ ]:
import os
from PIL import Image, UnidentifiedImageError

def mask_and_resize_batch(input_dir, output_dir, box, size=(224, 224)):
    """
    ディレクトリ内のすべての画像を作成日時順（正確には更新日時順）で処理。
    指定領域以外黒く塗りつぶして224x224へリサイズして保存。
    壊れた画像ファイルはスキップします。
    """

    os.makedirs(output_dir, exist_ok=True)

    # 画像ファイル一覧を収集
    files = [
        f for f in os.listdir(input_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))
    ]

    # 更新日時（mtime）でソート
    files.sort(key=lambda f: os.path.getmtime(os.path.join(input_dir, f)))

    for fname in files:
        in_path = os.path.join(input_dir, fname)
        out_path = os.path.join(output_dir, fname)

        try:
            img = Image.open(in_path).convert("RGB")
        except UnidentifiedImageError:
            print(f"⚠️ 壊れた画像をスキップ: {in_path}")
            continue
        except Exception as e:
            print(f"⚠️ 画像読み込み時にエラー発生 ({fname}): {e}")
            continue

        # 背景を黒で生成
        masked = Image.new("RGB", img.size, (0, 0, 0))

        # 指定領域をコピー
        region = img.crop(box)
        masked.paste(region, box)

        # リサイズして保存
        masked = masked.resize(size, Image.BICUBIC)
        masked.save(out_path)

        print(f"✅ Processed: {fname}")

    print("🎉 全画像の処理が完了しました！（作成日時順）")

if __name__ == "__main__":
    input_dir = "/home/jetson/jetracer/notebooks/camera/run-202511141612_cam1/xy/"
    output_dir = "/home/jetson/jetracer/notebooks/cropData/run-202511141612_cam1/xy/"
    #direction
    #box = (0, 70, 224, 150)
    #540
    box = (0, 45, 224, 224)

    mask_and_resize_batch(input_dir, output_dir, box)
